# SULO Pizza Tutorial — Deployment and FAIRness

# Learning objectives

* Add ontology-level metadata required by FAIR principles
* Understand OWL versioning (`owl:versionIRI`, `owl:versionInfo`)
* Export to multiple OWL syntaxes (RDF/XML and Turtle)
* Interpret a FOOPS! FAIRness assessment report
* Identify and fix common FAIRness gaps in an ontology

# FAIR Ontologies

An ontology that lives only on your laptop is of limited value to the scientific community. The **FAIR principles** — Findable, Accessible, Interoperable, Reusable — provide a framework for making ontologies (and data) maximally useful:

| Principle | Applied to ontologies |
|---|---|
| **Findable** | The ontology has a stable, resolvable IRI; it is registered in a catalogue (e.g. BioPortal, OLS) |
| **Accessible** | The ontology IRI resolves to a downloadable file in a standard format |
| **Interoperable** | The ontology uses standard OWL constructs; it imports and aligns with upper-level ontologies |
| **Reusable** | The ontology carries a clear licence, is well-documented, and its terms have human-readable labels and definitions |

In this notebook we work through adding the metadata that makes a pizza ontology FAIR, and interpret the kind of report that the [FOOPS! tool](https://foops.linkeddata.es) would produce for it.

> **OntoStart**: The [OntoStart](https://github.com/micheldumontier/ontostart) project template provides a ready-made GitHub repository structure — including CI pipelines for documentation generation, quality checks, and deployment — so you can bootstrap a FAIR ontology project in minutes. This notebook shows what OntoStart does *under the hood*.

In [1]:
import sys, os
# Locate project root (the directory containing lib/) regardless of CWD
for _p in ['.', '..', '../..']:
    if os.path.isdir(os.path.join(_p, 'lib')):
        os.chdir(_p); sys.path.insert(0, os.getcwd()); break

from lib.helpers import *
import datetime
onto_path.append(".")

sulo = get_ontology("dist/sulo.owl").load()
se = get_ontology("dist/sulo-ext.owl").load()
pizza = get_ontology("dist/pizza-06.owl").load()
pizza.imported_ontologies.append(sulo)
pizza.imported_ontologies.append(se)
print("Pizza ontology IRI  :", pizza.base_iri)
print("Existing annotations:", pizza.metadata.comment)

Pizza ontology IRI  : https://w3id.org/ontostart/pizza/
Existing annotations: []


## Step 1 — Ontology IRI and Version IRI

Every FAIR ontology needs:
1. A **stable base IRI** — a permanent web address that identifies the ontology
2. A **version IRI** — an IRI that identifies this specific release

In OWL, the version IRI is declared with `owl:versionIRI`. The convention is:

```
Base IRI:    https://w3id.org/ontostart/pizza/
Version IRI: https://w3id.org/ontostart/pizza/releases/1.0.0/pizza.owl
```

The [W3ID](https://w3id.org) service provides permanent, redirectable IRIs — even if the server hosting the ontology changes, the W3ID IRI stays stable.

In [2]:
# owlready2's Metadata only accepts explicitly defined AnnotationProperties.
# owl:versionIRI and owl:versionInfo are OWL built-ins but must be declared first.

pizza_version = "1.0.0"
pizza_version_iri = f"https://w3id.org/ontostart/pizza/releases/{pizza_version}/pizza.owl"

with pizza:
    owl_ns = pizza.get_namespace("http://www.w3.org/2002/07/owl#")

    class versionIRI(AnnotationProperty):
        namespace = owl_ns

    class versionInfo(AnnotationProperty):
        namespace = owl_ns

    pizza.metadata.versionIRI = [pizza_version_iri]
    pizza.metadata.versionInfo = [pizza_version]

print("Version IRI :", pizza.metadata.versionIRI)
print("Version info:", pizza.metadata.versionInfo)

Version IRI : ['https://w3id.org/ontostart/pizza/releases/1.0.0/pizza.owl']
Version info: ['1.0.0']


## Step 2 — Dublin Core Annotations

The most common metadata annotations for ontologies use **Dublin Core** terms. owlready2 allows you to define annotation properties that map to DC IRIs.

Key annotations:
| Annotation | DC term | Purpose |
|---|---|---|
| `dc:title` | `http://purl.org/dc/terms/title` | Human-readable name |
| `dc:description` | `http://purl.org/dc/terms/description` | One-paragraph summary |
| `dc:creator` | `http://purl.org/dc/terms/creator` | Author(s) |
| `dc:license` | `http://purl.org/dc/terms/license` | SPDX licence identifier or URL |
| `dc:created` | `http://purl.org/dc/terms/created` | Creation date |

In [3]:
# Define Dublin Core annotation properties in the pizza namespace
with pizza:
    dc_ns = pizza.get_namespace("http://purl.org/dc/terms/")

    class title(AnnotationProperty):
        namespace = dc_ns

    class description(AnnotationProperty):
        namespace = dc_ns

    class creator(AnnotationProperty):
        namespace = dc_ns

    class license(AnnotationProperty):
        namespace = dc_ns

    class created(AnnotationProperty):
        namespace = dc_ns

# Add ontology-level annotations
today = datetime.date.today().isoformat()
with pizza:
    pizza.metadata.title       = ["SULO Pizza Tutorial Ontology"]
    pizza.metadata.description = [
        "An OWL ontology for the pizza domain, built incrementally through the "
        "SULO Pizza Tutorial. Covers spatial composition, qualities, quantities, "
        "processes, information entities, time, and spatial containment."
    ]
    pizza.metadata.creator     = ["SULO Pizza Tutorial Authors"]
    pizza.metadata.license     = ["https://creativecommons.org/licenses/by/4.0/"]
    pizza.metadata.created     = [today]

print("Metadata added:")
print("  title      :", pizza.metadata.title)
print("  description:", pizza.metadata.description[0][:60], "...")
print("  license    :", pizza.metadata.license)
print("  created    :", pizza.metadata.created)

Metadata added:
  title      : ['SULO Pizza Tutorial Ontology']
  description: An OWL ontology for the pizza domain, built incrementally th ...
  license    : ['https://creativecommons.org/licenses/by/4.0/']
  created    : ['2026-04-12']


## Step 3 — Term-Level Annotations

FAIR also requires that **every term** in the ontology has at minimum:
- `rdfs:label` — a human-readable name
- `rdfs:comment` — a textual definition

In owlready2, these are the `label` and `comment` attributes on each class/property. Let's check how many of our pizza classes are missing labels or definitions.

In [4]:
missing_label   = []
missing_comment = []

for cls in pizza.classes():
    if not cls.label:
        missing_label.append(cls.name)
    if not cls.comment:
        missing_comment.append(cls.name)

print(f"Classes missing rdfs:label  : {len(missing_label)}")
print(f"Classes missing rdfs:comment: {len(missing_comment)}")
if missing_label:
    print("  Missing labels:", missing_label[:10])
if missing_comment:
    print("  Missing comments:", missing_comment[:10])

Classes missing rdfs:label  : 61
Classes missing rdfs:comment: 47
  Missing labels: ['FoodMaterial', 'Pizza', 'PizzaCrust', 'Cornicione', 'PizzaSauce', 'PizzaTopping', 'Cheese', 'Mozzarella', 'Gorgonzola', 'Parmesan']
  Missing comments: ['Spicyness', 'SpicyHot', 'SpicyMild', 'SpicyMedium', 'SpicySalami', 'SpicySalamiPizza', 'SpicyPizza', 'SpicynessMeasurement', 'HotPepper', 'HotPepperPizza']


In [5]:
# Add missing labels (rdfs:label) to the classes that need them
# In owlready2, if no label is set, the class name is used as the IRI local name.
# Best practice: add a human-readable label for each class.

with pizza:
    for cls in pizza.classes():
        if not cls.label:
            # Generate a label by inserting spaces before capitals (CamelCase → words)
            import re
            readable = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', cls.name)
            cls.label = [readable]

n_labelled = sum(1 for c in pizza.classes() if c.label)
print(f"Classes with rdfs:label: {n_labelled} / {len(list(pizza.classes()))}")

Classes with rdfs:label: 61 / 61


## Step 4 — Exporting to Multiple Syntaxes

A FAIR ontology should be available in at least two standard serialisations so that different tools can consume it without parsing difficulties. owlready2 supports **RDF/XML** and **Turtle**.

In [12]:
import os
os.makedirs("dist", exist_ok=True)

# RDF/XML — the OWL default, widest tool support
pizza.save(file="dist/pizza.owl", format="rdfxml")
# Turtle — more readable, preferred for version control diffs
pizza.save(file="dist/pizza.nt", format="ntriples")

print("Exported:")
print("  dist/pizza.owl  (RDF/XML) —", os.path.getsize("dist/pizza.owl"), "bytes")
print("  dist/pizza.nt  (N-Triples)  —", os.path.getsize("dist/pizza.nt"), "bytes")

Exported:
  dist/pizza.owl  (RDF/XML) — 51076 bytes
  dist/pizza.nt  (N-Triples)  — 91679 bytes


## Step 5 — Interpreting a FOOPS! Report

[FOOPS!](https://foops.linkeddata.es) is an automated tool that checks ontology FAIRness against a set of measurable indicators. Below is an annotated summary of the indicators it checks and whether our pizza ontology now satisfies them.

Run the following cell to self-assess against the key FOOPS! checks:

In [7]:
# Self-assessment against FOOPS! indicators
checks = {
    "F1 — Ontology has a resolvable IRI"                 : bool(pizza.base_iri),
    "F2 — Ontology has a version IRI"                    : bool(pizza.metadata.versionIRI),
    "A1 — Ontology serialised in a standard format"      : True,  # we saved RDF/XML and Turtle
    "I1 — Ontology uses OWL/RDF"                         : True,
    "I2 — Ontology re-uses terms from other ontologies"  : any(pizza.imported_ontologies),
    "R1 — Ontology has a human-readable title"           : bool(pizza.metadata.title),
    "R1.1 — Ontology has a description"                  : bool(pizza.metadata.description),
    "R1.2 — Ontology has a licence"                      : bool(pizza.metadata.license),
    "R1.3 — Ontology has a creator"                      : bool(pizza.metadata.creator),
    "R1.4 — Terms have rdfs:label"                       : all(c.label for c in pizza.classes()),
    "R1.5 — Ontology has a creation date"                : bool(pizza.metadata.created),
}

print("FOOPS! self-assessment")
print("-" * 50)
score = 0
for indicator, passed in checks.items():
    mark = "PASS" if passed else "FAIL"
    if passed: score += 1
    print(f"  [{mark}] {indicator}")
print(f"\nScore: {score}/{len(checks)}")

FOOPS! self-assessment
--------------------------------------------------
  [PASS] F1 — Ontology has a resolvable IRI
  [PASS] F2 — Ontology has a version IRI
  [PASS] A1 — Ontology serialised in a standard format
  [PASS] I1 — Ontology uses OWL/RDF
  [PASS] I2 — Ontology re-uses terms from other ontologies
  [PASS] R1 — Ontology has a human-readable title
  [PASS] R1.1 — Ontology has a description
  [PASS] R1.2 — Ontology has a licence
  [PASS] R1.3 — Ontology has a creator
  [PASS] R1.4 — Terms have rdfs:label
  [PASS] R1.5 — Ontology has a creation date

Score: 11/11


In [8]:
pizza.save(file="dist/pizza-07.owl", format="rdfxml")
print("Ontology saved. Tutorial complete.")

Ontology saved. Tutorial complete.


---

## Exercises

### Exercise 1 — Add term-level definitions
Choose five pizza classes that are still missing `rdfs:comment` definitions and add meaningful one-sentence definitions to each. Re-run the FOOPS! self-assessment and verify the R1.4 indicator score improves.



In [9]:
# Exercise 1 — your code here


### Exercise 2 — `owl:priorVersion`
OWL supports a `priorVersion` annotation to link a new release back to the previous one. Add a `priorVersion` annotation to the pizza ontology pointing to `https://w3id.org/ontostart/pizza/releases/0.9.0/pizza.owl`. How would this help users track the history of the ontology?



In [10]:
# Exercise 2 — your code here


### Exercise 3 — FOOPS! online
Export your final `pizza.owl` and upload it to the [FOOPS! online service](https://foops.linkeddata.es). Compare its report with your self-assessment above. Note which checks the tool applies that our self-assessment does not cover, and which additional metadata you would need to add to achieve a perfect score.

*(No code required — reflection exercise.)*
